<a href="https://colab.research.google.com/github/ihsann2410010447/2410010447_AkbarIhsanNugraha/blob/main/M7_Praktikum_Boston_Housing_Akbar_Ihsan_Nugraha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 Modul 7 — Praktikum Distribusi Normal (Gaussian)

### Studi Kasus: Boston Housing — Normal, CLT, Z-score & Weight Initialization

**Mata Kuliah:** Statistika & Probabilitas (TIF1501)
**Program Studi:** Teknik Informatika (55201) · FTI · Uniska MAB Banjarmasin
**Semester:** 5 · 3 SKS

---

## 🎯 Tujuan Pembelajaran

1. **Memahami** distribusi Normal N(μ, σ²), PDF, dan 68-95-99.7 rule.
2. **Membuktikan** Central Limit Theorem secara empiris dengan bootstrap.
3. **Menerapkan** Z-score standardization + StandardScaler untuk ML preprocessing.
4. **Mengimplementasikan** Xavier/He weight initialization pada MLP regression.

---

## 📋 Alur Notebook

| Bagian | Isi | Waktu |
|--------|-----|-------|
| **A** | Setup & Load Boston Housing | 10 menit |
| **B** | EDA Distribusi Fitur | 10 menit |
| **C** | Normal Fit untuk fitur rm | 10 menit |
| **D** | Verifikasi 68-95-99.7 Rule | 10 menit |
| **E** | Central Limit Theorem Demo | 15 menit |
| **F** | Z-score & StandardScaler | 10 menit |
| **G** | MLP dengan Xavier/He Init | 15 menit |
| **H** | Refleksi & Submit | 10 menit |

**Total: ± 90 menit**

---

## 💡 Kenapa Boston Housing?

Dataset klasik ML dari Kaggle:
- ✅ **506 rumah** di Boston dengan 13 fitur numerik lengkap
- ✅ Fitur `rm` (jumlah kamar) kuasi-Normal — bagus untuk fit Gaussian
- ✅ Fitur `medv` (target) skewed-right — demo CLT dari non-Normal
- ✅ Range skala berbeda (crim 0-89, tax 187-711) — demo StandardScaler
- ✅ Cocok untuk regression → MLP sederhana


---

# 🔧 A. Setup & Load Boston Housing

## A.1 Import Library


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scipy untuk distribusi Normal
from scipy import stats
from scipy.stats import norm, shapiro

# sklearn untuk standardization + MLP
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Library siap untuk Modul 7!")
print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")


## A.2 Load Dataset dari Kaggle Mirror


In [ ]:
# URL dataset Boston Housing
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"

df = pd.read_csv(url)
print(f"Dataset berhasil dimuat!")
print(f"Shape: {df.shape[0]} baris × {df.shape[1]} kolom")
print(f"\nKolom:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")
df.head()


---

# 🔍 B. EDA: Distribusi Fitur

## B.1 Histogram Semua Fitur Numerik


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for i, col in enumerate(df.columns):
    ax = axes[i // 5, i % 5]
    df[col].hist(bins=30, ax=ax, color='#3B82F6', edgecolor='black', alpha=0.75)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel("")
# Hide extra subplot
if len(df.columns) < 15:
    axes[-1, -1].axis('off')
plt.tight_layout()
plt.show()


## B.2 Skewness Test


In [ ]:
skew_summary = pd.DataFrame({
    'mean': df.mean().round(3),
    'std': df.std().round(3),
    'skewness': df.skew().round(3),
    'bentuk': df.skew().apply(lambda s:
        'Kuasi-Normal' if abs(s) < 0.5
        else 'Skewed-right' if s > 0 else 'Skewed-left'),
})
print(skew_summary)
print("\n💡 Fitur 'rm' paling mendekati Normal → dipakai untuk Section C.")
print("💡 Fitur 'medv' (target) skewed → dipakai untuk CLT di Section E.")


---

# 📊 C. Normal Fit untuk fitur `rm`

**Tujuan Pembelajaran 1** — Fit Normal distribution ke data riil.

## C.1 Estimasi Parameter μ, σ (MLE)


In [ ]:
rm = df['rm']
mu_hat = rm.mean()
sigma_hat = rm.std(ddof=0)  # MLE: pakai population std

print(f"Estimasi parameter untuk rm:")
print(f"  μ̂ = mean    = {mu_hat:.4f}")
print(f"  σ̂ = std     = {sigma_hat:.4f}")
print(f"  N (sample)  = {len(rm)}")

# Formal normality test (Shapiro-Wilk)
stat_sw, p_sw = shapiro(rm)
print(f"\nShapiro-Wilk test:")
print(f"  W = {stat_sw:.4f}")
print(f"  p = {p_sw:.4e}")
print(f"  Interpretasi: {'Normal' if p_sw > 0.05 else 'Bukan Normal sempurna (tapi tetap bell-shaped)'}")


## C.2 Plot PDF Overlay Histogram


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

# Histogram empiris
ax.hist(rm, bins=30, density=True, color='#3B82F6', edgecolor='black',
        alpha=0.6, label=f'rm empiris (n={len(rm)})')

# Overlay Normal PDF teoritis
x_range = np.linspace(rm.min(), rm.max(), 300)
pdf_vals = norm.pdf(x_range, loc=mu_hat, scale=sigma_hat)
ax.plot(x_range, pdf_vals, 'r-', linewidth=2.5,
        label=f'N(μ={mu_hat:.2f}, σ={sigma_hat:.2f})')

# Mark μ
ax.axvline(mu_hat, color='orange', linestyle='--', linewidth=2,
           label=f'μ = {mu_hat:.2f}')
ax.axvspan(mu_hat - sigma_hat, mu_hat + sigma_hat, alpha=0.15, color='orange',
           label=f'±1σ: {mu_hat-sigma_hat:.2f} s/d {mu_hat+sigma_hat:.2f}')

ax.set_xlabel('rm (avg rooms per dwelling)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Fit Normal pada fitur rm (Boston Housing)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


## C.3 QQ-Plot untuk Validasi Visual


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
stats.probplot(rm, dist="norm", plot=ax)
ax.set_title('QQ-Plot: rm vs Normal theoretical quantiles', fontsize=13)
ax.get_lines()[0].set_markerfacecolor('#3B82F6')
ax.get_lines()[0].set_markeredgecolor('black')
ax.get_lines()[1].set_color('red')
ax.get_lines()[1].set_linewidth(2)
plt.tight_layout()
plt.show()

print("\n💡 Points dekat garis merah → distribusi mendekati Normal.")
print("   Deviasi di ekor → tail behavior sedikit berbeda dari Normal murni.")


---

# 📏 D. Verifikasi Aturan 68-95-99.7

**Tujuan Pembelajaran 1** — Verifikasi three-sigma rule secara empiris pada data riil.


In [ ]:
# Hitung proporsi empiris dalam ±1σ, ±2σ, ±3σ
within_1sigma = ((rm >= mu_hat - sigma_hat) & (rm <= mu_hat + sigma_hat)).mean()
within_2sigma = ((rm >= mu_hat - 2*sigma_hat) & (rm <= mu_hat + 2*sigma_hat)).mean()
within_3sigma = ((rm >= mu_hat - 3*sigma_hat) & (rm <= mu_hat + 3*sigma_hat)).mean()

print(f"Verifikasi 68-95-99.7 Rule pada rm:")
print(f"  ±1σ:  empiris = {within_1sigma*100:.2f}%  |  teori = 68.27%")
print(f"  ±2σ:  empiris = {within_2sigma*100:.2f}%  |  teori = 95.45%")
print(f"  ±3σ:  empiris = {within_3sigma*100:.2f}%  |  teori = 99.73%")

# Visualisasi
fig, ax = plt.subplots(figsize=(11, 6))
ax.hist(rm, bins=30, color='#3B82F6', edgecolor='black', alpha=0.6)

for k, label, color in [(1, '±1σ', 'orange'), (2, '±2σ', 'green'), (3, '±3σ', 'red')]:
    ax.axvline(mu_hat - k*sigma_hat, color=color, linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvline(mu_hat + k*sigma_hat, color=color, linestyle='--', linewidth=1.5, alpha=0.7)

ax.axvline(mu_hat, color='black', linestyle='-', linewidth=2, label=f'μ = {mu_hat:.2f}')
ax.set_xlabel('rm', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'Three-Sigma Rule pada rm (empiris: {within_1sigma*100:.0f}/{within_2sigma*100:.0f}/{within_3sigma*100:.0f}%)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


---

# 🌐 E. Central Limit Theorem Demonstrasi

**Tujuan Pembelajaran 2** — Buktikan secara empiris bahwa mean sampel dari distribusi SKEWED (medv) konvergen ke Normal saat n membesar.

## E.1 Visualisasi Target medv (parent — skewed)


In [ ]:
medv = df['medv']
print(f"medv statistik:")
print(f"  Mean     = {medv.mean():.3f}")
print(f"  Std      = {medv.std():.3f}")
print(f"  Skewness = {medv.skew():.3f}  (jauh dari 0 → skewed)")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(medv, bins=40, color='#F43F5E', edgecolor='black', alpha=0.75)
ax.axvline(medv.mean(), color='black', linestyle='--', linewidth=2, label=f'μ = {medv.mean():.2f}')
ax.set_xlabel('medv (median home value in $1000)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'Distribusi medv — SKEWED-RIGHT (bukan Normal!)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


## E.2 Bootstrap Mean untuk n = 5, 30, 100


In [ ]:
np.random.seed(42)
n_simulations = 2000
sample_sizes = [5, 30, 100]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n in zip(axes, sample_sizes):
    sample_means = [medv.sample(n=n, replace=True).mean()
                    for _ in range(n_simulations)]
    sample_means = np.array(sample_means)

    # Histogram sample means
    ax.hist(sample_means, bins=35, density=True, color='#8B5CF6',
            edgecolor='black', alpha=0.75)

    # Overlay Normal berdasarkan CLT prediction
    mu_pred = medv.mean()
    se_pred = medv.std() / np.sqrt(n)  # SE = σ/√n
    x = np.linspace(sample_means.min(), sample_means.max(), 200)
    ax.plot(x, norm.pdf(x, mu_pred, se_pred), 'r-', linewidth=2.5,
            label=f'N({mu_pred:.1f}, {se_pred:.2f}²)')

    ax.set_title(f'n = {n}\n(mean of sample means: {sample_means.mean():.2f})', fontsize=12)
    ax.set_xlabel('X̄ (sample mean)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=10)

plt.suptitle(f'CLT: Mean sampel dari medv (skewed) konvergen ke Normal', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Semakin besar n, distribusi X̄ semakin mendekati Normal — meski parent skewed!")
print("💡 Standard error SE = σ/√n: makin kecil dengan n membesar.")


---

# 🎯 F. Z-score Standardization

**Tujuan Pembelajaran 3** — Apply z-transform manual + StandardScaler.

## F.1 Z-score Manual untuk Outlier Detection


In [ ]:
# Hitung z-score manual untuk fitur crim (crime rate)
crim = df['crim']
mu_crim, sigma_crim = crim.mean(), crim.std()

z_scores = (crim - mu_crim) / sigma_crim

print(f"Z-score untuk crim:")
print(f"  Mean(z)  = {z_scores.mean():.4f}  (expected ≈ 0)")
print(f"  Std(z)   = {z_scores.std():.4f}   (expected ≈ 1)")

# Outlier detection (|z| > 3)
outliers = df[abs(z_scores) > 3]
print(f"\n🚨 Outlier (|z| > 3): {len(outliers)} baris")
print("Top 5 outlier dengan crim tertinggi:")
print(outliers.nlargest(5, 'crim')[['crim', 'medv']].round(2))


## F.2 StandardScaler dengan sklearn


In [ ]:
# Pisahkan fitur dan target
X = df.drop('medv', axis=1)
y = df['medv']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit scaler di train, transform ke keduanya
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Sebelum StandardScaler (X_train):")
print(f"  Mean per kolom (sample):\n{X_train.mean().round(2).head()}")
print(f"  Std per kolom (sample) :\n{X_train.std().round(2).head()}")

print("\nSetelah StandardScaler (X_train_scaled):")
print(f"  Mean per kolom (semua) : {X_train_scaled.mean(axis=0).round(4)}")
print(f"  Std per kolom (semua)  : {X_train_scaled.std(axis=0).round(4)}")

print("\n💡 Mean ≈ 0, Std ≈ 1 untuk semua fitur — siap untuk ML!")


---

# 🧠 G. MLP Regression dengan Xavier/He Init

**Tujuan Pembelajaran 4** — Train MLP regressor dengan weight init berbeda, bandingkan konvergensi.

## G.1 Baseline MLP tanpa standardization


In [ ]:
# Tanpa scaling → biasanya lambat / tidak konvergen baik
mlp_raw = MLPRegressor(hidden_layer_sizes=(32, 16), activation='relu',
                      solver='adam', max_iter=200, random_state=0, early_stopping=False)
mlp_raw.fit(X_train, y_train)
pred_raw = mlp_raw.predict(X_test)
r2_raw = r2_score(y_test, pred_raw)
rmse_raw = np.sqrt(mean_squared_error(y_test, pred_raw))

print(f"MLP tanpa scaling:")
print(f"  R² test   = {r2_raw:.4f}")
print(f"  RMSE test = {rmse_raw:.3f}")


## G.2 MLP dengan Standardization (data normal input + He-init via sklearn default)

sklearn `MLPRegressor` dengan ReLU menggunakan **Glorot/He-style initialization** by default. Kombinasi dengan `StandardScaler` → input mendekati N(0,1), konvergensi jauh lebih stabil.


In [ ]:
mlp_scaled = MLPRegressor(hidden_layer_sizes=(32, 16), activation='relu',
                         solver='adam', max_iter=200, random_state=0, early_stopping=False)
mlp_scaled.fit(X_train_scaled, y_train)
pred_scaled = mlp_scaled.predict(X_test_scaled)
r2_scaled = r2_score(y_test, pred_scaled)
rmse_scaled = np.sqrt(mean_squared_error(y_test, pred_scaled))

print(f"MLP dengan StandardScaler:")
print(f"  R² test   = {r2_scaled:.4f}")
print(f"  RMSE test = {rmse_scaled:.3f}")

print(f"\n🎯 Improvement karena input Normal:")
print(f"   R² : {r2_raw:.3f} → {r2_scaled:.3f}  (+{(r2_scaled-r2_raw):.3f})")
print(f"   RMSE: {rmse_raw:.2f} → {rmse_scaled:.2f}  ({(rmse_scaled-rmse_raw):+.2f})")


## G.3 Visualisasi: Loss Curve Perbandingan


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(mlp_raw.loss_curve_, color='#F43F5E', linewidth=2, label='Tanpa scaling')
axes[0].plot(mlp_scaled.loss_curve_, color='#10B981', linewidth=2, label='Dengan StandardScaler')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Training loss', fontsize=12)
axes[0].set_title('Loss Curve: Impact of Normal-ish Input', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].set_yscale('log')

# Scatter predicted vs actual
axes[1].scatter(y_test, pred_raw, color='#F43F5E', alpha=0.5, label=f'Tanpa scaling (R²={r2_raw:.3f})')
axes[1].scatter(y_test, pred_scaled, color='#10B981', alpha=0.5, label=f'Scaled (R²={r2_scaled:.3f})')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', linewidth=1.5)
axes[1].set_xlabel('Actual medv', fontsize=12)
axes[1].set_ylabel('Predicted medv', fontsize=12)
axes[1].set_title('Predicted vs Actual (Test Set)', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\n💡 Standardization membuat loss landscape lebih bulat → GD konvergen cepat & stabil.")


### ✏️ Jawaban Anda:

**1. Normal Distribution (Distribusi Normal):**
* **Mengapa hampir Normal?** Jumlah kamar (`rm`) cenderung mengikuti distribusi Normal karena merupakan fenomena fisik dan sosial yang alami. Mayoritas rumah dibangun dengan jumlah kamar rata-rata (sekitar 5-6 kamar) karena kebutuhan pasar dan standar konstruksi umum, sehingga terkonsentrasi di tengah (bell-shaped).
* **Penyebab deviasi di ekor:** Deviasi di ekor (outlier) disebabkan oleh adanya kasus ekstrem di dunia nyata. Misalnya, rumah mewah/apartemen besar dengan jumlah kamar sangat banyak (ekor kanan) atau hunian mikro/sekat sangat sempit dengan kamar sangat sedikit (ekor kiri) yang tidak mencerminkan populasi perumahan standar kelas menengah.

**2. Three-sigma Rule (Aturan 68-95-99.7):**
* **Apakah persis sama?** Tidak, proporsi empiris tidak persis sama tetapi sangat mendekati nilai teoretis.
* **Mengapa berbeda sedikit?**
  1. *Ukuran sampel terbatas* ($N = 506$), sedangkan teori probabilitas mengasumsikan sampel mendekati tak hingga ($N \to \infty$).
  2. *Data asli tidak murni Gaussian* (memiliki skewness minor dan outlier), sedangkan teori 68-95-99.7 diturunkan secara matematis dari fungsi densitas Normal murni tanpa noise.

**3. Central Limit Theorem (CLT):**
* **Mengapa bisa menghasilkan sample means yang Normal?** Berdasarkan Central Limit Theorem (CLT), distribusi dari rata-rata sampel (sample means) akan mendekati distribusi Normal seiring bertambahnya ukuran sampel ($n$), tidak peduli apa pun bentuk distribusi populasinya (bahkan jika sangat skewed seperti `medv`). Batas ukuran sampel $n \ge 30$ secara empiris sudah cukup besar untuk meredam kecondongan (skewness) populasi asal.
* **Contoh aplikasi di Data Science:** A/B Testing. Saat kita membandingkan rata-rata metrik (seperti *Click-Through Rate* atau durasi sesi) antara kelompok kontrol dan eksperimen, kita menggunakan uji hipotesis (seperti t-test) yang mengasumsikan distribusi rata-rata sampel adalah Normal, berkat CLT.

**4. StandardScaler:**
* **Mengapa wajib fit hanya di X_train?** Proses `fit` menghitung nilai rata-rata ($\mu$) dan standar deviasi ($\sigma$) dari data. Kita hanya boleh menghitung parameter ini dari `X_train` agar model tidak mengetahui informasi apa pun terkait distribusi data uji (`X_test`).
* **Risiko jika salah (melakukan fit pada X_test atau seluruh dataset):** Terjadi kebocoran data (*Data Leakage*). Evaluasi performa model pada data uji menjadi terlalu optimis (overestimasi) dan bias, karena model secara tidak langsung sudah mendeteksi parameter statistik dari data uji sebelum proses testing dilakukan.

**5. Weight Initialization (Inisialisasi Bobot):**
* **Pilihan:** **He Initialization (He Normal / He Uniform)**.
* **Penjelasan:** Xavier initialization dirancang dengan asumsi fungsi aktivasi linier atau aktif di sekitar nol (seperti tanh atau sigmoid). Namun, fungsi **ReLU** memotong semua nilai negatif menjadi nol ($f(x) = \max(0, x)$) yang berarti setengah dari neuron akan mati secara statistik di setiap layer. He Initialization mengoreksi hal ini dengan mengalikan variansi bobot dengan faktor $2/n_{in}$ (dua kali lipat dari Xavier) untuk menjaga agar variansi sinyal dan gradien tidak lenyap (*vanishing gradient*) atau meledak (*exploding gradient*) selama proses *forward* dan *backward propagation* pada arsitektur Deep Learning.

## H.2 Checklist Submit

Sebelum submit, pastikan:

- [ ] Semua cell code berhasil dijalankan tanpa error
- [ ] Fit Normal pada `rm` sudah menghasilkan μ ≈ 6.28, σ ≈ 0.70
- [ ] Verifikasi 68-95-99.7 menunjukkan proporsi dekat teori
- [ ] CLT bootstrap untuk n=5, 30, 100 muncul 3 panel
- [ ] StandardScaler menghasilkan mean ≈ 0, std ≈ 1
- [ ] MLP dengan scaling R² > tanpa scaling
- [ ] 5 pertanyaan refleksi sudah dijawab
- [ ] Notebook di-rename: `M7-Praktikum-Normal-NamaAnda.ipynb`
- [ ] Upload ke GitHub repository Anda
- [ ] Link GitHub dikumpulkan via LMS Uniska

## H.3 Push ke GitHub

**Melalui Web:**
1. Buka repository `statistika-probabilitas-uniska` di GitHub Anda.
2. Klik folder `notebooks/`, lalu **Add file → Upload files**.
3. Drag & drop file notebook.
4. Commit message: `"Add Modul 7: Distribusi Normal & CLT"`.
5. Klik **Commit changes**.

---

## 🎓 Selamat! Anda Berhasil...

✅ Fit distribusi Normal ke data riil Boston Housing
✅ Verifikasi aturan 68-95-99.7 secara empiris
✅ Membuktikan CLT dengan bootstrap mean dari distribusi skewed
✅ Apply StandardScaler untuk ML preprocessing
✅ Train MLP regressor dengan weight init yang tepat

## 📚 Bacaan Lanjutan

- **Bruce, P.** (2020). *Practical Statistics for Data Scientists*, Chapter 2.7-2.9 (Normal & CLT).
- **Glorot, X. & Bengio, Y.** (2010). *Understanding the difficulty of training deep feedforward neural networks* (Xavier init paper).
- **He, K. et al.** (2015). *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification* (He init paper).
- **StatQuest YouTube** — "Normal Distribution Clearly Explained" dan "Central Limit Theorem".

## 🔮 Persiapan Modul 8 & UTS

Modul 8 adalah **Review & UTS CPMK032** (bobot 20%):
- Review Modul 1-6 (M1 Data, M2 Prob, M3 Bayes, M4 RV, M5 Diskret, M7 Normal)
- Solving 10 soal UTS tahun lalu bersama
- Konsultasi project Kaggle

**Sebelum UTS:**
1. **QUIZ 2** (bobot 6%) di akhir Modul 7 — materi: Modul 5 + 6
2. Selesaikan Modul-7.ipynb, push ke GitHub
3. Baca Bruce Ch. 2.7-2.9
4. Buat cheatsheet pribadi untuk Modul 1-6

---

*Notebook by: Prodi Teknik Informatika · Uniska MAB Banjarmasin · Edisi 2026*
